# Explorar dados da Câmara
Lê os Parquet direto do repositório. Rode as células em ordem.

In [ ]:
!git clone -q --depth 1 https://github.com/cristianojoseblanco/cenarioPoliticoBR.git repo 2>/dev/null || git -C repo pull -q
%cd repo
!pip -q install duckdb pyyaml

In [ ]:
import duckdb
con = duckdb.connect()
V = "read_parquet('data/staging/camara/votacoesVotos/**/*.parquet', hive_partitioning=true)"
VT = "read_parquet('data/staging/camara/votacoes/**/*.parquet', hive_partitioning=true)"
con.sql(f'SELECT ano, count(*) AS votos, count(DISTINCT idVotacao) AS votacoes, count(DISTINCT deputado_id) AS deputados FROM {V} GROUP BY 1 ORDER BY 1').df()

In [ ]:
# Distribuição dos tipos de voto
con.sql(f'SELECT voto, count(*) AS n FROM {V} GROUP BY 1 ORDER BY 2 DESC').df()

In [ ]:
# Votações nominais por órgão (Plenário vs. comissões)
con.sql(f'''SELECT v.siglaOrgao, count(DISTINCT v.id) AS votacoes_nominais
FROM {VT} v JOIN (SELECT DISTINCT idVotacao FROM {V}) x ON x.idVotacao = v.id
GROUP BY 1 ORDER BY 2 DESC''').df()

In [ ]:
# Taxa de voto 'Sim' por partido
con.sql(f'''SELECT deputado_siglaPartido AS partido, count(*) AS votos,
  round(100.0*avg(CASE WHEN voto='Sim' THEN 1 ELSE 0 END),1) AS pct_sim
FROM {V} GROUP BY 1 ORDER BY votos DESC''').df()